In [ ]:
# load data
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import math
from PIL import Image
import os

labels = pd.read_csv('01_data/01_processed/postcard_fine_labels.csv')
meta = pd.read_csv('01_data/00_metadata/20230301-Postcards.csv')

meta = meta.rename(columns={
    'Color 340$o_standardized': 'color_orig',
    'Resolver URL 856$u':        'resolver_url',
    'Publisher 264$b_clean':     'publisher'
})
meta['IE_id']  = meta['resolver_url'].str.extract(r'/(IE\d+)/')
labels['IE_id'] = labels['image_path'].str.extract(r'/(IE\d+)/')

merged = labels.merge(meta[['IE_id', 'color_orig', 'publisher']], on='IE_id', how='left')
print(f"Total images: {len(merged)}")
print(merged['fine_label'].value_counts())

In [ ]:
# Post-processing: move monotone images out of color group
# Move images with monotone metadata labels that were
# incorrectly classified into the color group back to monotone categories.
# Only images that are BOTH (a) metadata-labelled as Blue/Green/Purple/Red
# AND (b) machine-classified as color_photo or color_handcolored are moved.

color_labels = ['color_photo', 'color_handcolored']

merged.loc[(merged['color_orig'] == 'Blue')   & (merged['fine_label'].isin(color_labels)), 'fine_label'] = 'monotone_blue'
merged.loc[(merged['color_orig'] == 'Green')  & (merged['fine_label'].isin(color_labels)), 'fine_label'] = 'monotone_green'
merged.loc[(merged['color_orig'] == 'Purple') & (merged['fine_label'].isin(color_labels)), 'fine_label'] = 'monotone_purple'
merged.loc[(merged['color_orig'] == 'Red')    & (merged['fine_label'].isin(color_labels)), 'fine_label'] = 'monotone_red'

print("After post-processing:")
print(merged['fine_label'].value_counts())

In [ ]:
# Merge dark/light subgroups into single bw and sepia categories
merged['fine_label'] = merged['fine_label'].replace({
    'bw_dark':    'bw',
    'bw_light':   'bw',
    'sepia_dark': 'sepia',
    'sepia_light':'sepia',
})

print("Final label distribution:")
print(merged['fine_label'].value_counts())

# Save
merged.drop(columns=['color_orig', 'publisher', 'IE_id']).to_csv(
    '01_data/01_processed/postcard_fine_labels.csv',
    index=False
)
print("\nSaved.")

In [ ]:
# Sample images per final label
def show_samples(df, label, n=6, random_state=42):
    subset = df[df['fine_label'] == label]
    if len(subset) == 0:
        print(f"{label}: no images")
        return
    sample = subset.sample(min(n, len(subset)), random_state=random_state)
    cols = 3
    rows = math.ceil(len(sample) / cols)
    fig, axes = plt.subplots(rows, cols, figsize=(12, 4 * rows))
    axes = np.array(axes).reshape(-1)
    for ax in axes:
        ax.axis('off')
    for ax, (_, row) in zip(axes, sample.iterrows()):
        try:
            img = Image.open(row['image_path']).convert('RGB')
            ax.imshow(img)
            ax.set_title(os.path.basename(row['image_path']), fontsize=7)
        except Exception:
            ax.text(0.5, 0.5, 'load error', ha='center', va='center')
        ax.axis('off')
    plt.suptitle(f'fine_label = {label}  (n={len(subset)})', fontsize=13)
    plt.tight_layout()
    plt.show()

df = pd.read_csv('01_data/01_processed/postcard_fine_labels.csv')
for label in sorted(df['fine_label'].unique()):
    show_samples(df, label)